# Lab 19: The Disappearing Difference

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-19.ipynb)

**What you will do:** time how long it takes you to spot the one change in a flicker change-blindness demo, compare your detection times with the range Lab 19 reports, and run a pixel-difference script that finds the same change instantly, to see exactly what your attention system is not doing.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 19 you watched two nearly identical photographs alternate with a blank screen between
them (Rensink et al., 1997) and tried to spot the one large change. It commonly took ten
seconds or more, even though the change, once seen, looked obvious. This is **change
blindness**: the brain stores a sparse, attention-limited model of a scene rather than a
pixel-by-pixel copy, so a change is invisible unless your attention happens to be on the
changing region at the exact moment the disruption (the blank screen) masks the motion signal
that would otherwise give it away.

## Record your results

Time how long it takes you to spot the change in a few different flicker change-blindness
demonstrations (search for more than one online example), and note whether the change was in
the centre of the scene or off to the side, and whether it was a colour change, an object
appearing or disappearing, or a position change.

In [ ]:
# example data: replace with your own stopwatch timings, in seconds
trials = pd.DataFrame({
    "demo": ["demo 1", "demo 2", "demo 3", "demo 4"],
    "change_location": ["central", "peripheral", "central", "peripheral"],
    "change_type": ["object removed", "colour change", "position change", "object removed"],
    "detection_time_s": [8.5, 22.0, 6.0, 31.5],
})
display(trials)
trials.plot.bar(x="demo", y="detection_time_s", color="steelblue", figsize=(6, 3), legend=False)
plt.ylabel("detection time (s)"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 19 reports that spotting the change "commonly" takes "ten seconds or more, and sometimes
much longer." It also discusses, in the AI Connection, that peripheral or subtle changes are
typically harder to catch than central, obvious ones. Compare your own detection times with
that description, and check whether your central-change trials were faster than your
peripheral ones.

In [ ]:
print(f"Mean detection time: {trials['detection_time_s'].mean():.1f} s "
      f"(range {trials['detection_time_s'].min():.1f}-{trials['detection_time_s'].max():.1f} s)")
slow_trials = (trials["detection_time_s"] >= 10).sum()
print(f"{slow_trials} of {len(trials)} trials took 10 seconds or more, matching the lab's"
      f" 'commonly ten seconds or more' description.")

by_location = trials.groupby("change_location")["detection_time_s"].mean()
print("\nMean detection time by location:")
print(by_location)
if "central" in by_location and "peripheral" in by_location:
    if by_location["central"] < by_location["peripheral"]:
        print("Central changes were faster to spot than peripheral ones, as the text's discussion of")
        print("attention and gaze position would predict.")
    else:
        print("Peripheral changes were not slower here -- with only a few trials this is easily due to")
        print("chance, or to a peripheral change that happened to be unusually large or high-contrast.")

## The AI side

The lab contrasts your own attention-limited change detection with a computer's brute-force
pixel comparison, which "will instantly detect any difference, no matter how small." Below, a
simple pixel-difference script does exactly that on two near-identical images, finding the
changed region in a fraction of a second, something that took you many seconds to do with
your own eyes.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import cv2
from PIL import Image, ImageDraw

# Build two near-identical synthetic scenes (in place of the two lab photographs, since we
# cannot bundle copyrighted stimulus images with this notebook).
def make_scene(rect_present):
    img = Image.new("RGB", (300, 200), (235, 235, 235))
    draw = ImageDraw.Draw(img)
    draw.ellipse((30, 30, 90, 90), fill=(180, 120, 60))     # a "tree"
    draw.rectangle((150, 60, 180, 160), fill=(90, 90, 200)) # a "building"
    if rect_present:
        draw.rectangle((220, 40, 270, 90), fill=(200, 60, 60))  # the object that will disappear
    return np.array(img)

scene_a = make_scene(rect_present=True)
scene_b = make_scene(rect_present=False)

diff = cv2.absdiff(scene_a, scene_b)
diff_gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(diff_gray, 25, 255, cv2.THRESH_BINARY)
ys, xs = np.where(mask > 0)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(scene_a); axes[0].set_title("scene A"); axes[0].axis("off")
axes[1].imshow(scene_b); axes[1].set_title("scene B"); axes[1].axis("off")
axes[2].imshow(mask, cmap="gray"); axes[2].set_title("pixel difference"); axes[2].axis("off")
plt.tight_layout(); plt.show()

if len(xs):
    print(f"Changed region found instantly: x = {xs.min()}-{xs.max()}, y = {ys.min()}-{ys.max()}")
print("Compare this near-zero detection time with your own stopwatch results above.")

The pixel-difference script finds the change in a single pass because it never compresses the
scene the way your visual system does: it keeps every pixel and compares all of them. That is
also its weakness: the lab's autonomous-driving example notes that this same brute-force
approach can still miss an adversarial sticker that subtly alters a road sign, because a
learned object-recognition model, like your brain, assimilates small deviations from what it
"expects" a sign to look like. Pixel comparison and human attention fail in different ways:
one misses subtle, meaningful changes it was never told to look for; the other misses large,
meaningless ones it was not attending to.

## Pool the class data

Pool the class's detection times, split by whether the change was central or peripheral, to
check the pattern with more data (Rensink et al., 1997, found peripheral and less
"interesting" changes take longer to detect).

In [ ]:
import io
csv_text = """id,change_location,detection_time_s
P01,central,7.5
P01,peripheral,18.0
P02,central,9.0
P02,peripheral,25.5
P03,central,5.5
P03,peripheral,15.0
P04,central,11.0
P04,peripheral,30.0
"""  # example data: replace with your class CSV, e.g. pd.read_csv("changeblindness_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
display(class_df.groupby("change_location")["detection_time_s"].agg(["mean", "std", "count"]))

boots = {}
for loc, group in class_df.groupby("change_location"):
    boot = [group["detection_time_s"].sample(len(group), replace=True).mean() for _ in range(3000)]
    boots[loc] = np.percentile(boot, [2.5, 97.5])
for loc, (lo, hi) in boots.items():
    print(f"{loc}: 95% bootstrap interval {lo:.1f}-{hi:.1f} s")

class_df.boxplot(column="detection_time_s", by="change_location", grid=False, figsize=(5, 4))
plt.suptitle(""); plt.title("Class detection times"); plt.ylabel("seconds"); plt.show()

## Questions to think about

1. The pixel-difference script finds the change with zero attention and zero delay, while you needed several seconds. The lab argues this trade-off, comprehensive but expensive versus selective but efficient, "recurs throughout AI design." Where else in this chapter (or the book) have you seen a system choose selective, efficient processing over comprehensive, expensive processing?
2. Rensink et al. (1997), cited in the lab, is the classic source for the flicker change-blindness paradigm. Based on your own detection times, does the size of the change (rate it 1-5 for how large the changed object was) predict how fast you found it, or does location (central/peripheral) seem to matter more?
3. The Beyond the Lab section compares gradual, undetected change to climate change and institutional erosion, arguing that "the antidote is not better vigilance... but better record-keeping." Relate this directly to what the pixel-difference script does differently from your own eyes.
4. The lab's road-sign example says an AI model can fail at "semantic" change detection, missing an adversarial sticker, even though it never fails at literal pixel comparison. Why can a system be perfect at one kind of change detection and vulnerable at another?

## Challenge

Try Step 3's real-world version with a friend: look away while they change one thing among
several objects on a table. Time how long it takes you to spot the change, and note how many
objects were on the table. Repeat with more objects. Does your detection time increase with
the number of objects, the way you would expect if change blindness is fundamentally about
limited attention rather than limited eyesight?

In [ ]:
# Example data: replace with your own real-world (table-objects) trials.
n_objects = [3, 5, 7, 9]                    # example data: how many objects were on the table
detection_time_s = [3.0, 6.5, 12.0, 19.5]   # example data: how long it took you to spot the change

plt.figure(figsize=(5, 3.5))
plt.plot(n_objects, detection_time_s, "o-", color="firebrick")
plt.xlabel("number of objects on the table"); plt.ylabel("detection time (s)"); plt.show()
print("If detection time rises with the number of objects, this supports the lab's account: change")
print("blindness comes from a fixed amount of attention spread over more things, not from a fixed")
print("inability to see change.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-19.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")